2.3.1. Импорты, seed и устройство

In [14]:
# Базовые библиотеки
import os
import random
import numpy as np
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn

# Датасеты / трансформации
import torchvision
import torchvision.datasets 
import torchvision.transforms

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


2.3.2. Данные и DataLoader

In [15]:
from torchvision.transforms import ToTensor
from torch.utils.data import random_split, DataLoader
import os
import torch

transform = ToTensor()

cifar_root = "./data"

train_dataset = torchvision.datasets.CIFAR10(
    root=cifar_root, train=True, download=False, transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root=cifar_root, train=False, download=False, transform=transform
)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

g = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size], generator=g)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

xb, yb = next(iter(train_loader))
print("batch_size:", train_loader.batch_size, "xb.shape:", xb.shape, "yb.shape:", yb.shape)
print("min:", xb.min().item(), "max:", xb.max().item())

batch_size: 128 xb.shape: torch.Size([128, 3, 32, 32]) yb.shape: torch.Size([128])
min: 0.0 max: 1.0


2.3.3. Модель MLP и цикл обучения

In [16]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)

model = MLP().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy(preds, targets):
    predicted_classes = preds.argmax(dim=1)
    correct = (predicted_classes == targets).sum().item()
    return correct / len(targets)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_acc = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(logits, y)
    return total_loss / len(loader), total_acc / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_acc = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item()
            total_acc += accuracy(logits, y)
    return total_loss / len(loader), total_acc / len(loader)


num_epochs = 10

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f}")
    print(f"Val   loss: {val_loss:.4f} | Val   acc: {val_acc:.4f}")
    print("-" * 40)

Epoch 1/10
Train loss: 1.8955 | Train acc: 0.3141
Val   loss: 1.7494 | Val   acc: 0.3711
----------------------------------------
Epoch 2/10
Train loss: 1.6998 | Train acc: 0.3932
Val   loss: 1.6638 | Val   acc: 0.4119
----------------------------------------
Epoch 3/10
Train loss: 1.6281 | Train acc: 0.4167
Val   loss: 1.5964 | Val   acc: 0.4326
----------------------------------------
Epoch 4/10
Train loss: 1.5762 | Train acc: 0.4382
Val   loss: 1.5839 | Val   acc: 0.4327
----------------------------------------
Epoch 5/10
Train loss: 1.5320 | Train acc: 0.4545
Val   loss: 1.5303 | Val   acc: 0.4578
----------------------------------------
Epoch 6/10
Train loss: 1.4989 | Train acc: 0.4636
Val   loss: 1.5157 | Val   acc: 0.4572
----------------------------------------
Epoch 7/10
Train loss: 1.4732 | Train acc: 0.4729
Val   loss: 1.5143 | Val   acc: 0.4650
----------------------------------------
Epoch 8/10
Train loss: 1.4431 | Train acc: 0.4840
Val   loss: 1.5284 | Val   acc: 0.4532
-

In [ ]:
import json
import pandas as pd

class ConfigurableMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes, use_batchnorm: bool = False, dropout_p: float = 0.0, num_classes: int = 10):
        super().__init__()
        layers = [nn.Flatten()]
        in_features = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(in_features, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p > 0.0:
                layers.append(nn.Dropout(dropout_p))
            in_features = h
        layers.append(nn.Linear(in_features, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def run_experiment(
    experiment_id: str,
    model_cfg: dict,
    optim_cfg: dict,
    num_epochs: int,
    use_early_stopping: bool = False,
    patience: int = 5,
):
    set_seed(model_cfg.get("seed", 42))

    model = ConfigurableMLP(
        input_dim=3 * 32 * 32,
        hidden_sizes=model_cfg["hidden_sizes"],
        use_batchnorm=model_cfg.get("batchnorm", False),
        dropout_p=model_cfg.get("dropout_p", 0.0),
        num_classes=10,
    ).to(device)

    if optim_cfg["optimizer"] == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=optim_cfg["lr"],
            weight_decay=optim_cfg.get("weight_decay", 0.0),
        )
    elif optim_cfg["optimizer"] == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=optim_cfg["lr"],
            momentum=optim_cfg.get("momentum", 0.0),
            weight_decay=optim_cfg.get("weight_decay", 0.0),
        )
    else:
        raise ValueError("Unknown optimizer")

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_state_dict = None
    best_epoch = 0
    epochs_trained = 0
    no_improve = 0

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        epochs_trained += 1

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_state_dict = model.state_dict()
            best_epoch = epoch + 1
            no_improve = 0
        else:
            no_improve += 1

        print(f"[{experiment_id}] Epoch {epoch+1}/{num_epochs} | "
              f"Train loss {train_loss:.4f}, acc {train_acc:.4f} | "
              f"Val loss {val_loss:.4f}, acc {val_acc:.4f}")

        if use_early_stopping and no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    run_info = {
        "experiment_id": experiment_id,
        "dataset": "CIFAR10",
        "seed": model_cfg.get("seed", 42),
        "model_summary": f"hidden={model_cfg['hidden_sizes']}, "
                          f"bn={model_cfg.get('batchnorm', False)}, "
                          f"dropout={model_cfg.get('dropout_p', 0.0)}",
        "optimizer": optim_cfg["optimizer"],
        "lr": optim_cfg["lr"],
        "momentum": optim_cfg.get("momentum", 0.0),
        "weight_decay": optim_cfg.get("weight_decay", 0.0),
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
    }

    return model, history, run_info, best_state_dict

runs = []
artifacts_dir = "artifacts"
figures_dir = os.path.join(artifacts_dir, "figures")
os.makedirs(figures_dir, exist_ok=True)

base_model_cfg = {
    "hidden_sizes": [512, 256, 128],
    "batchnorm": False,
    "dropout_p": 0.0,
    "seed": 42,
}
base_optim_cfg = {"optimizer": "Adam", "lr": 1e-3, "weight_decay": 0.0}

model_E1, hist_E1, run_E1, _ = run_experiment("E1", base_model_cfg, base_optim_cfg, num_epochs=15)
runs.append(run_E1)

model_cfg_E2 = {**base_model_cfg, "dropout_p": 0.3}
model_E2, hist_E2, run_E2, _ = run_experiment("E2", model_cfg_E2, base_optim_cfg, num_epochs=15)
runs.append(run_E2)

model_cfg_E3 = {**base_model_cfg, "batchnorm": True, "dropout_p": 0.0}
model_E3, hist_E3, run_E3, _ = run_experiment("E3", model_cfg_E3, base_optim_cfg, num_epochs=15)
runs.append(run_E3)

best_reg_run = max([run_E2, run_E3], key=lambda r: r["best_val_accuracy"])
if best_reg_run["experiment_id"] == "E2":
    best_model_cfg = model_cfg_E2
else:
    best_model_cfg = model_cfg_E3

patience = 5
model_E4, hist_E4, run_E4, best_state_E4 = run_experiment(
    "E4", best_model_cfg, base_optim_cfg, num_epochs=30, use_early_stopping=True, patience=patience
)
runs.append(run_E4)

best_model_path = os.path.join(artifacts_dir, "best_model.pt")
torch.save(best_state_E4, best_model_path)

best_config = {
    "experiment_id": "E4",
    "dataset": "CIFAR10",
    "seed": best_model_cfg.get("seed", 42),
    "model_cfg": best_model_cfg,
    "optimizer_cfg": base_optim_cfg,
    "early_stopping": {"patience": patience, "best_epoch": run_E4["best_epoch"]},
}
with open(os.path.join(artifacts_dir, "best_config.json"), "w") as f:
    json.dump(best_config, f, indent=2)

optim_cfg_O1 = {"optimizer": "Adam", "lr": 1e-1, "weight_decay": 0.0}
model_O1, hist_O1, run_O1, _ = run_experiment("O1", best_model_cfg, optim_cfg_O1, num_epochs=8)
runs.append(run_O1)

optim_cfg_O2 = {"optimizer": "Adam", "lr": 1e-5, "weight_decay": 0.0}
model_O2, hist_O2, run_O2, _ = run_experiment("O2", best_model_cfg, optim_cfg_O2, num_epochs=8)
runs.append(run_O2)

optim_cfg_O3 = {"optimizer": "SGD", "lr": 1e-2, "momentum": 0.9, "weight_decay": 1e-4}
model_O3, hist_O3, run_O3, _ = run_experiment("O3", best_model_cfg, optim_cfg_O3, num_epochs=15)
runs.append(run_O3)

runs_df = pd.DataFrame(runs)
runs_df.to_csv(os.path.join(artifacts_dir, "runs.csv"), index=False)

plt.figure(figsize=(8, 5))
plt.plot(hist_E4["train_loss"], label="train_loss")
plt.plot(hist_E4["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "curves_best.png"))
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(hist_O1["train_loss"], label="O1 train_loss (lr=1e-1)")
plt.plot(hist_O2["train_loss"], label="O2 train_loss (lr=1e-5)")
plt.xlabel("epoch")
plt.ylabel("train loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "curves_lr_extremes.png"))
plt.close()

best_model = ConfigurableMLP(
    input_dim=3 * 32 * 32,
    hidden_sizes=best_model_cfg["hidden_sizes"],
    use_batchnorm=best_model_cfg.get("batchnorm", False),
    dropout_p=best_model_cfg.get("dropout_p", 0.0),
    num_classes=10,
).to(device)

best_model.load_state_dict(best_state_E4)

test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
print(f"Best model (E4) on TEST: loss={test_loss:.4f}, acc={test_acc:.4f}")

[E1] Epoch 1/15 | Train loss 1.9271, acc 0.2900 | Val loss 1.7481, acc 0.3674
[E1] Epoch 2/15 | Train loss 1.7257, acc 0.3781 | Val loss 1.6694, acc 0.3988
[E1] Epoch 3/15 | Train loss 1.6418, acc 0.4109 | Val loss 1.6580, acc 0.4062
[E1] Epoch 4/15 | Train loss 1.5845, acc 0.4309 | Val loss 1.5768, acc 0.4346
[E1] Epoch 5/15 | Train loss 1.5270, acc 0.4525 | Val loss 1.5207, acc 0.4594
[E1] Epoch 6/15 | Train loss 1.4881, acc 0.4683 | Val loss 1.5219, acc 0.4639
[E1] Epoch 7/15 | Train loss 1.4503, acc 0.4828 | Val loss 1.4832, acc 0.4719
[E1] Epoch 8/15 | Train loss 1.4290, acc 0.4891 | Val loss 1.4707, acc 0.4759
[E1] Epoch 9/15 | Train loss 1.4021, acc 0.4980 | Val loss 1.4594, acc 0.4758
[E1] Epoch 10/15 | Train loss 1.3700, acc 0.5095 | Val loss 1.4537, acc 0.4805
[E1] Epoch 11/15 | Train loss 1.3491, acc 0.5168 | Val loss 1.4101, acc 0.4985
[E1] Epoch 12/15 | Train loss 1.3257, acc 0.5242 | Val loss 1.4604, acc 0.4778
[E1] Epoch 13/15 | Train loss 1.3030, acc 0.5344 | Val loss 1